# Quantum harmonic oscillator: ladders and coherent states

**Professor Nano Labs · Quantum foundations**

**Try it live in your browser:** [Harmonic oscillator lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/harmonic-oscillator/)
**Watch first:** [Solving the quantum harmonic oscillator with ladder operators](https://www.youtube.com/watch?v=jHjebJtBXUs)
**Watch later:** [Time independent Schrodinger equation and stationary states](https://www.youtube.com/watch?v=cm8tf6NuePI) · [Schrodinger equation and time evolution of quantum state](https://www.youtube.com/watch?v=gXkbSp2xzlc) · [How exactly does trapped ions perform quantum computing?](https://www.youtube.com/watch?v=5qpXdGSLWig)

A particle in the potential $\tfrac12m\omega^2x^2$. We measure position in $x_0=\sqrt{\hbar/m\omega}$, energy in $\hbar\omega$ and time in periods $T=2\pi/\omega$. Then
$$H=a^\dagger a+\tfrac12,\qquad x=\frac{a+a^\dagger}{\sqrt2},\qquad p=\frac{i(a^\dagger-a)}{\sqrt2},\qquad [a,a^\dagger]=1.$$
The three sections match the lab's three tabs. It needs only NumPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import factorial

N = 60                                            # rungs kept
a = np.diag(np.sqrt(np.arange(1, N)), 1)          # lowering operator: a|n> = sqrt(n)|n-1>
ad = a.T                                          # raising operator
X = (a + ad) / np.sqrt(2)
P = 1j * (ad - a) / np.sqrt(2)
Hm = ad @ a + 0.5 * np.eye(N)

comm = a @ ad - ad @ a
print('[a, a+] = 1 on every rung but the last (cut off):', np.allclose(comm[:-1, :-1], np.eye(N - 1)))
print('energies:', np.linalg.eigvalsh(Hm)[:6])

def psi_table(xs, nmax):
    # psi_n(x) by the stable recurrence psi_{n+1} = sqrt(2/(n+1)) x psi_n - sqrt(n/(n+1)) psi_{n-1}
    T = [np.pi ** -0.25 * np.exp(-xs ** 2 / 2)]
    T.append(np.sqrt(2) * xs * T[0])
    for n in range(1, nmax - 1):
        T.append(np.sqrt(2 / (n + 1)) * xs * T[n] - np.sqrt(n / (n + 1)) * T[n - 1])
    return np.array(T)

xs = np.linspace(-10, 10, 2001)
T = psi_table(xs, N)
h = np.polynomial.hermite.hermval(xs, [0] * 5 + [1]) * np.exp(-xs ** 2 / 2) / np.sqrt(2 ** 5 * factorial(5) * np.sqrt(np.pi))
print('psi_5 from the recurrence matches the Hermite formula:', np.allclose(T[5], h))

## 1 · Climb the ladder

Each rung $|n\rangle$ has energy $n+\tfrac12$ and a wavefunction with $n$ nodes. Written with a derivative, $a^\dagger=(\xi-d/d\xi)/\sqrt2$ turns $\psi_{n-1}$ into $\sqrt n\,\psi_n$. At high rungs the probability density follows the classical one, $1/\pi\sqrt{A^2-x^2}$, on average.

In [ ]:
dx = xs[1] - xs[0]
for n in (1, 4, 10):
    raised = (xs * T[n - 1] - np.gradient(T[n - 1], dx)) / np.sqrt(2)
    print(f'(x - d/dx) psi_{n-1} / sqrt2 vs sqrt({n}) psi_{n}: max difference {np.max(np.abs(raised - np.sqrt(n) * T[n])):.1e}')
lowered = (xs * T[0] + np.gradient(T[0], dx)) / np.sqrt(2)
print('a psi_0 by derivatives, largest value:', np.max(np.abs(lowered)).round(6))

nodes = lambda f: int(np.sum(np.abs(np.diff(np.sign(f[np.abs(f) > 1e-8]))) > 0))
print('nodes of psi_0 .. psi_6:', [nodes(T[n]) for n in range(7)])

n = 30; A = np.sqrt(2 * n + 1)
inside = np.abs(xs) < A
plt.figure(figsize=(8, 3.2))
plt.plot(xs, T[n] ** 2, label='quantum |psi_30|^2')
plt.plot(xs[inside], 1 / (np.pi * np.sqrt(A ** 2 - xs[inside] ** 2)), '--', label='classical')
plt.ylim(0, 0.25); plt.xlim(-10, 10); plt.xlabel('x / x0'); plt.legend(); plt.show()

## 2 · Superpositions move

Each amplitude turns as $e^{-i(n+1/2)\omega t}$. A single rung never moves; two rungs beat at their energy difference. Position links only neighbouring rungs, so $\langle x\rangle$ swings only for $|n-m|=1$.

In [ ]:
def evolve(c, t):
    return c * np.exp(-2j * np.pi * t * (np.arange(N) + 0.5))

def expect(op, c):
    return np.real(np.vdot(c, op @ c))

ts = np.linspace(0, 2, 401)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.2))
for ax, (m, k) in zip(axs, [(0, 1), (0, 2)]):
    c = np.zeros(N, complex); c[m] = c[k] = 1 / np.sqrt(2)
    xm = [expect(X, evolve(c, t)) for t in ts]
    sx = [np.sqrt(expect(X @ X, evolve(c, t)) - expect(X, evolve(c, t)) ** 2) for t in ts]
    ax.plot(ts, xm, label='<x>'); ax.plot(ts, sx, '--', label='width')
    ax.set(title=f'|{m}> + |{k}>', xlabel='t / T', ylabel='x / x0'); ax.legend()
plt.tight_layout(); plt.show()

## 3 · Coherent states

$|\alpha\rangle=e^{-|\alpha|^2/2}\sum_n\alpha^n/\sqrt{n!}\,|n\rangle$ is the ground state moved off centre. It satisfies $a|\alpha\rangle=\alpha|\alpha\rangle$, its quanta follow a Poisson distribution, and in time it stays a Gaussian of width $1/\sqrt2$ while its centre follows the classical orbit. A Gaussian of any other width is a squeezed state: it breathes twice per period.

In [ ]:
def coherent(alpha):
    n = np.arange(N)
    return np.exp(-abs(alpha) ** 2 / 2) * alpha ** n / np.sqrt([float(factorial(k)) for k in n])

def gaussian(xc, pc, sigma):
    # project a Gaussian packet onto the rungs
    g = (2 * np.pi * sigma ** 2) ** -0.25 * np.exp(-(xs - xc) ** 2 / (4 * sigma ** 2) + 1j * pc * xs)
    return T @ g * dx

alpha = 2.0
c = coherent(alpha)
print('a|alpha> = alpha|alpha> (away from the cut-off):', np.allclose((a @ c)[:40], alpha * c[:40]))
print('P(3), P(4):', np.abs(c[3]) ** 2, np.abs(c[4]) ** 2)
for t in (0, 0.25, 0.4):
    ct = evolve(c, t)
    sx = np.sqrt(expect(X @ X, ct) - expect(X, ct) ** 2); sp = np.sqrt(expect(P @ P, ct) - expect(P, ct) ** 2)
    print(f't = {t}: <x> = {expect(X, ct):+.3f} (classical {np.sqrt(2) * alpha * np.cos(2 * np.pi * t):+.3f}), dx = {sx:.3f}, dx*dp = {sx * sp:.3f}')

sq = gaussian(2 * np.sqrt(2), 0, 0.5 / np.sqrt(2))
widths = [np.sqrt(expect(X @ X, evolve(sq, t)) - expect(X, evolve(sq, t)) ** 2) for t in ts]
plt.figure(figsize=(6, 3)); plt.plot(ts, widths); plt.xlabel('t / T'); plt.ylabel('width / x0'); plt.title('a squeezed state breathes'); plt.show()

## Check the lab's answers

These asserts confirm the numbers in the lab's "Predict first" answers.

In [ ]:
def spread(c):
    return np.sqrt(expect(X @ X, c) - expect(X, c) ** 2), np.sqrt(expect(P @ P, c) - expect(P, c) ** 2)
ground = np.zeros(N); ground[0] = 1
# Q1: a|0> = 0, E0 = 1/2, dx*dp = 1/2
assert np.allclose(a @ ground, 0) and np.isclose(np.prod(spread(ground)), 0.5)
# Q2: a+|3> = 2|4>; psi_4 has four nodes
three = np.zeros(N); three[3] = 1
assert np.allclose(ad @ three, 2 * np.eye(N)[4]) and nodes(T[4]) == 4
# Q3: turning point of rung 30 at sqrt(61) = 7.81
assert np.isclose(np.sqrt(61), 7.81, atol=0.005)
# Q4: |2> is stationary; its phase turns once every 2T/5
two = np.eye(N)[2].astype(complex)
assert all(np.allclose(np.abs(evolve(two, t)), np.abs(two)) for t in ts[::50])
assert np.isclose(np.angle(evolve(two, 0.4)[2]) % (2 * np.pi), 0, atol=1e-9)
# Q5: |0> + |1> swings with amplitude 1/sqrt2
c01 = (np.eye(N)[0] + np.eye(N)[1]) / np.sqrt(2)
assert np.isclose(max(expect(X, evolve(c01, t)) for t in ts), 1 / np.sqrt(2))
# Q6: |0> + |2> keeps <x> = 0 and breathes between 0.89 and 1.49
c02 = (np.eye(N)[0] + np.eye(N)[2]) / np.sqrt(2)
w = [spread(evolve(c02, t))[0] for t in ts]
assert max(abs(expect(X, evolve(c02, t))) for t in ts) < 1e-12 and np.isclose(min(w), 0.890, atol=1e-3) and np.isclose(max(w), 1.486, atol=1e-3)
# Q7: |alpha| = 2: swing 2.83, width 0.71 always, <n> = 4, P(3) = P(4) = 0.195
assert all(np.isclose(spread(evolve(c, t))[0], 1 / np.sqrt(2)) for t in ts[::40])
assert np.isclose(expect(ad @ a, c), 4) and np.isclose(abs(c[3]) ** 2, 0.1954, atol=1e-4) and np.isclose(abs(c[4]) ** 2, abs(c[3]) ** 2)
# Q8: squeezed to half width: dx from 0.35 to 1.41, dx*dp up to 1.06
prods = [np.prod(spread(evolve(sq, t))) for t in ts]
assert np.isclose(min(widths), 0.354, atol=1e-3) and np.isclose(max(widths), 1.414, atol=1e-3) and np.isclose(max(prods), 1.062, atol=2e-3)
print('All checks passed.')

## Things to try

1. **Anharmonic spacing.** Add $\lambda x^4$ to the Hamiltonian, diagonalise it in the $N$-rung basis, and evolve a coherent state. Its rungs are no longer equally spaced: watch it spread and later partly reassemble.
2. **A displaced number state.** Move $|1\rangle$ off centre instead of the ground state. Does it keep its shape as it swings?
3. **Thermal light.** Mix the rungs with Boltzmann weights. How does the number distribution differ from a coherent state with the same $\langle n\rangle$?
4. **Real units.** For a $^{40}$Ca$^+$ ion at $\omega/2\pi=1$ MHz, convert $x_0$ and $\hbar\omega$ to nm and neV. How many quanta make a swing of 1 μm?